In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\Ashish yadav\anaconda3\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
409,I really really love this show! I have always ...,positive
494,"To be frank, this is probably the best version...",positive
579,"Wow, what an overrated movie this turned out t...",negative
555,The name (Frau) of the main character is the G...,negative
655,This is only the second time I stopped a video...,negative


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
import nltk 
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to C:\Users\Ashish
[nltk_data]     yadav\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Ashish
[nltk_data]     yadav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
409,really really love show always liked s show sp...,positive
494,frank probably best version book sound movie v...,positive
579,wow overrated movie turned be supposed an extr...,negative
555,name frau main character german word woman kno...,negative
655,second time stopped video dvd part way through...,negative


In [7]:
df['sentiment'].value_counts()

sentiment
positive    257
negative    243
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
409,really really love show always liked s show sp...,1
494,frank probably best version book sound movie v...,1
579,wow overrated movie turned be supposed an extr...,0
555,name frau main character german word woman kno...,0
655,second time stopped video dvd part way through...,0


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/ASHishYADAav2003/MLOPs_Prometheus_Grafana_capstone_project.mlflow')
dagshub.init(repo_owner='ASHishYADAav2003', repo_name='MLOPs_Prometheus_Grafana_capstone_project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as ASHishYADAav2003

Initialized MLflow to track repo "ASHishYADAav2003/MLOPs_Prometheus_Grafana_capstone_project"

Repository ASHishYADAav2003/MLOPs_Prometheus_Grafana_capstone_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/0066e92829224761b69bd33da4da061a', creation_time=1777579591151, experiment_id='0', last_update_time=1777579591151, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-01 17:18:02,334 - INFO - Starting MLflow run...
2026-05-01 17:18:03,584 - INFO - Logging preprocessing parameters...
2026-05-01 17:18:04,833 - INFO - Initializing Logistic Regression model...
2026-05-01 17:18:04,833 - INFO - Fitting the model...
2026-05-01 17:18:04,899 - INFO - Model training complete.
2026-05-01 17:18:04,899 - INFO - Logging model parameters...
2026-05-01 17:18:05,311 - INFO - Making predictions...
2026-05-01 17:18:05,311 - INFO - Calculating evaluation metrics...
2026-05-01 17:18:05,336 - INFO - Logging evaluation metrics...
2026-05-01 17:18:07,018 - INFO - Saving and logging the model...
2026/05/01 17:18:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 17:18:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run invincible-chimp-198 at: https://dagshub.com/ASHishYADAav2003/MLOPs_Prometheus_Grafana_capstone_project.mlflow/#/experiments/0/runs/75cf4d5570b9472cbf27a70075e5f57a
🧪 View experiment at: https://dagshub.com/ASHishYADAav2003/MLOPs_Prometheus_Grafana_capstone_project.mlflow/#/experiments/0
